# Cython developer guide and experimentation

In [1]:
%load_ext cython

## Parallel neighbours getter

In [8]:
%%cython -a --compile-args=-fopenmp --link-args=-fopenmp

# distutils: define_macros=NPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION
# distutils: language = c++
# cython: wraparound = False
# cython: boundscheck = False
# cython: cdivision = True

import numpy as np
cimport numpy as np
from cython.parallel cimport prange, threadid

from commonnn cimport _types
from commonnn._primitive_types cimport AINDEX

from libc.stdlib cimport malloc, free


cdef AINDEX _get_n_neighbours(
        _types.InputDataExtInterface input_data,
        _types.NeighboursGetterExtInterface ngetter,
        _types.NeighboursExtInterface neighbours,
        _types.ClusterParameters params) except -1 nogil:

    cdef AINDEX n = input_data._n_points
    cdef AINDEX count = 0
    cdef AINDEX i

    for i in range(n):
        ngetter._get(i, input_data, neighbours, params)
        count += neighbours._get_n_points()

    return count


cdef AINDEX _get_n_neighbours_parallel(
        _types.InputDataExtInterface input_data,
        _types.NeighboursGetterExtInterface ngetter,
        _types.NeighboursExtInterface neighbours,
        _types.ClusterParameters params,
        AINDEX n_threads) except -1 nogil:

    cdef AINDEX n = input_data._n_points
    cdef AINDEX count = 0
    cdef AINDEX i
    cdef AINDEX tid

    for i in prange(n, num_threads=n_threads):
        tid = threadid()
        ngetter._get_p(i, input_data, neighbours, params, tid)
        count += neighbours._get_n_points_p(tid)

    return count


def get_n_neighbours(
        input_data: np.ndarray,
        radius: float = 0.1,
        parallel: bool = False,
        n_threads: int = 1,
        ):

    input_data_ = _types.InputDataExtComponentsMemoryview(input_data)
    params = _types.RadiusParameters.from_mapping({"radius_cutoff": radius})

    ngetter = _types.NeighboursGetterExtBruteForce(
        _types.DistanceGetterExtMetric(
            _types.MetricExtEuclidean()
        )
    )

    if parallel:
        neighbours = _types.NeighboursExtParallelVector(n_threads=n_threads)
        return _get_n_neighbours_parallel(input_data_, ngetter, neighbours, params, n_threads)
    else:
        neighbours = _types.NeighboursExtVector()
        return _get_n_neighbours(input_data_, ngetter, neighbours, params)

In [9]:
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

from commonnn import _types


noisy_circles, _ = datasets.make_circles(
    n_samples=2000,
    factor=.5,
    noise=.05
    )
noisy_circles = StandardScaler().fit_transform(noisy_circles)

In [10]:
get_n_neighbours(noisy_circles, radius=0.1)

25816

In [13]:
%timeit get_n_neighbours(noisy_circles, radius=0.1)

45.5 ms ± 79 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
get_n_neighbours(noisy_circles, radius=0.1, parallel=True)

25816

In [14]:
%timeit get_n_neighbours(noisy_circles, radius=0.1, parallel=True)

46.7 ms ± 4.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
for _ in range(10):
    print(get_n_neighbours(noisy_circles, radius=0.1, parallel=True, n_threads=2))

25816
25816
25816
25816
25816
25816
25816
25816
25816
25816


In [16]:
%timeit get_n_neighbours(noisy_circles, radius=0.1, parallel=True, n_threads=2)

24.6 ms ± 1.76 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
